# Procedimento para calcular população de bacias hidrossanitárias
##### Este documento define as etapas para a obtenção do número de habitantes inseridos na Área de Prestação de Serviços de bacias de esgotamento

Importação de bibliotecas:

In [79]:
import pandas as pd
import geopandas as gpd
import os

## 1º Passo: Importação dos dados
##### Setores Censitários: https://www.ibge.gov.br/estatisticas/sociais/trabalho/22827-censo-demografico-2022.html?edicao=41852&t=resultados 
Fazer download da malha de setores centiários por UF
##### Domicílios: https://www.ibge.gov.br/estatisticas/sociais/populacao/38734-cadastro-nacional-de-enderecos-para-fins-estatisticos.html?edicao=38891&t=resultados
Selecionar arquivos por município
##### APS encaminhado pela CORSAN; Bacias delimitadas pelo Analista. Coluna com o nome das bacias também deve ser especificado.

*Buscar código do município em https://www.ibge.gov.br/explica/codigos-dos-municipios.php

In [80]:
#Caçapava do Sul
setores = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul\Arquivos Baixados\RS_setores_CD2022.gpkg')
domicilios = pd.read_csv(r'C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul\Arquivos Baixados\4302808\4302808.csv',delimiter = ';')
aps = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul\Arquivos Criados\APS.gpkg')
bacias = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul\Arquivos Criados\BaciasH_v2.gpkg')
coluna_nome_bacias = 'Name'
caminho_exportacao = r'C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul\popdom2022_esp1e8_bh.xlsx'
crs = "EPSG:31982"

## Funções auxiliares

In [81]:
#Funções auxiliares

#contagem de domicílios em cada setor na APS

import geopandas as gpd

def somar_extensao_polig(lines_gdf, polys_gdf, poly_id_col="Nome"):
    """
    Retorna um DataFrame com o comprimento (m e km) das LINHAS contidas em cada polígono.
    Requer ambos em CRS projetado (unidades em METROS).
    - Filtra geometrias nulas/vazias
    - Valida polígonos (make_valid/buffer(0))
    - Faz overlay com keep_geom_type=False e filtra só linhas
    - Soma por polígono e adiciona coluna em km
    """
    # Cópias e colunas necessárias
    lines = lines_gdf[["geometry"]].copy()
    polys = polys_gdf[[poly_id_col, "geometry"]].copy()

    # CRS: alinhar se necessário
    if lines.crs != polys.crs:
        polys = polys.to_crs(lines.crs)

    # Limpeza: remover nulos/vazios
    lines = lines[lines.geometry.notnull() & ~lines.geometry.is_empty]
    polys = polys[polys.geometry.notnull() & ~polys.geometry.is_empty]

    # Validar polígonos (Shapely 2 -> make_valid; fallback buffer(0))
    try:
        polys["geometry"] = polys.geometry.make_valid()
    except Exception:
        polys["geometry"] = polys.buffer(0)

    # Overlay SEM restringir tipo de geometria
    inter = gpd.overlay(lines, polys, how="intersection", keep_geom_type=False)

    # Ficar só com partes lineares (descarta GeometryCollection/Polígonos/Pontos)
    inter = inter[inter.geom_type.isin(["LineString", "MultiLineString"])].copy()

    if inter.empty:
        # Retorno “vazio” com colunas esperadas
        return gpd.GeoDataFrame(
            {poly_id_col: [], "Extensão de Rede (m)": [], "Extensão de Rede (km)": []}
        )

    # Comprimento em metros (CRS deve estar em metros!)
    inter["Extensão de Rede (m)"] = inter.geometry.length

    # Soma por polígono
    out = (
        inter.groupby(poly_id_col, as_index=False)["Extensão de Rede (m)"]
        .sum()
        .sort_values(poly_id_col)
    )
    out["Extensão de Rede (km)"] = out["Extensão de Rede (m)"] / 1000.0
    return out

    return out

def contar_pontos_poligono(polygons, points, polygon_id_col="poly_id", predicate="intersects"):
    if polygons.crs != points.crs:
        points = points.to_crs(polygons.crs)

    # Garante coluna de ID
    if polygon_id_col not in polygons.columns:
        polygons = polygons.reset_index(drop=False).rename(columns={"index": polygon_id_col})

    joined = gpd.sjoin(points, polygons[[polygon_id_col, "geometry"]], predicate=predicate)
    counts = joined.groupby(polygon_id_col).size().rename("n_pontos").reset_index()
    
    result = polygons.merge(counts, on=polygon_id_col, how="left")
    result["n_pontos"] = result["n_pontos"].fillna(0).astype(int)
    
    return result

# Não é necessário mexer nisso abaixo

## 2º Passo: Tratamento dos dados

Conforme Diretriz Corsan (2025), o IBGE considera, para a densidade domiciliar, somente os domicílios particulares ocupados (v0007), o qual não representa a realidade das economias residenciais no cadastro da Corsan/Aegea. Portanto, deve-se recalcular a densidade domiciliar dos setores censitários. Para recalcular a densidade domiciliar, deve-se dividir a população (v0001) pelo total de domicílios particulares (v0003), gerando uma nova coluna “Densidade”.

In [82]:
setores['Densidade'] = setores['v0001']/setores['v0003']

É necessário filtrar os domicílios particulares (COD_ESPECIE = 1) e igrejas (COD_ESPECIE = 8) e transformar csv de domicílios em um arquivo georreferenciado

In [83]:
domparticular = domicilios[(domicilios['COD_ESPECIE'] == 1) |(domicilios['COD_ESPECIE'] == 8) ]
domparticular = gpd.GeoDataFrame(domparticular, geometry=gpd.points_from_xy(domparticular.LONGITUDE, domparticular.LATITUDE), crs="EPSG:4326")

Deve-se colocar tudo no mesmo CRS definido

In [84]:
domparticular = domparticular.to_crs(crs)
bacias = bacias.to_crs(crs)
aps = aps.to_crs(crs)
setores = setores.to_crs(crs) #convertendo pra sistema de coordenadas padrão

##### Intersecção entre setores e APS

In [85]:
setores_aps = gpd.clip(setores, aps) #interseção entre setores e APS
setores_aps = setores_aps[setores_aps['v0003']>0]
domparticular_aps = gpd.clip(domparticular, aps) #interseção entre domicílios e APS

## 3º Passo: Cálculo da população na APS
##### As etapas realizadas são:
- Intersecção entre Setores e APS
- Intersecção entre Domicílios e APS
- Contagem de domicílios em cada setor na APS
- Calculo da população

##### Contagem de domicílios em cada setor na APS

In [86]:
populacao_aps = contar_pontos_poligono(setores_aps, domparticular_aps)

##### Cálculo da população com a densidade e n_pontos criado

In [87]:
populacao_aps['População 2022'] = populacao_aps['Densidade']*populacao_aps['n_pontos']

pop = populacao_aps['População 2022'].sum()
print(f"A população total na APS em 2022 é de {pop:.0f}")
econ = len(domparticular_aps)
print(f"A população total na APS em 2022 é de {econ:.0f}")

A população total na APS em 2022 é de 26457
A população total na APS em 2022 é de 13439


## 4º Passo: Cálculo da população por bacia (2022)

Após delimitar as bacias para pelo menos 90% dos domicílios do IBGE (Censo 2022), deverá ser identificado quantos domicílios estão inseridos em cada bacia. As etapas realizadas são:
- Criação de camada com domicílios classificados por setor e bacia
- Criação de camada com bacias divididas em setores
- Calculo da população com base na densidade de cada domicílio dentro de cada setor dividido pela bacia
- Agrupamento dos valores por bacia, gerando a quantidade de população e domicílios por bacia

##### Intersecções entre domicílios, setores e bacias

In [88]:
camada_unida = gpd.sjoin(
    domparticular_aps,
    bacias, 
    predicate="intersects",
    how="left"
)
camada_unida = camada_unida.drop(columns=['index_right'], errors='ignore')

dompart_setores = gpd.sjoin(
    camada_unida,
    populacao_aps,  
    predicate="intersects",
    how="left"
)

bacias_setores = gpd.sjoin(
    populacao_aps,
    bacias,  
    predicate="intersects",
    how="left"
)

#dompart_setores_filtrado = dompart_setores[[coluna_nome_bacias, 'CD_SETOR']]
dompart_setores_filtrado = dompart_setores
#bacias_setores_filtrado = bacias_setores[[coluna_nome_bacias, 'CD_SETOR','Densidade']]
bacias_setores_filtrado = bacias_setores

In [89]:
#bacias_setores_filtrado.to_excel(os.path.join(r'C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul\bacias_setores_filtrado.xlsx'))
#dompart_setores_filtrado.to_excel(os.path.join(r'C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul\dompart_setores_filtrado.xlsx'))

##### Contagem da quantidade de vezes que uma combinação Setores Censitários + Bacia aparece

In [90]:
# Passo 1: Contar ocorrências de Nome + CD_SETOR na planilha de referência
contagem = (
    dompart_setores_filtrado
    .groupby([coluna_nome_bacias, 'CD_SETOR'])
    .size()
    .reset_index(name='Domicílios')
)

# Passo 2: Fazer merge com o DataFrame base
bacias_setores_filtrado = bacias_setores_filtrado.merge(contagem, on=[coluna_nome_bacias, 'CD_SETOR'], how='left')

# Passo 3: Substituir NaN por 0 (caso não tenha ocorrência)
bacias_setores_filtrado['Domicílios'] = bacias_setores_filtrado['Domicílios'].fillna(0).astype(int)

##### Cálculo da população por combinação Setores Censitários + Bacia

In [91]:
bacias_setores_filtrado['População'] = bacias_setores_filtrado['Domicílios']*bacias_setores_filtrado['Densidade']

##### Soma da população calculada por bacia

In [92]:
bacias_populacao = bacias_setores_filtrado[[coluna_nome_bacias,'Domicílios','População']].groupby(coluna_nome_bacias).sum()
bacias_populacao

,Domicílios,População
Name,,
Bacia 01,55,109.179775
Bacia 01.1,25,38.392857
Bacia 02,556,1223.547315
Bacia 02.1,1179,2630.597045
Bacia 02.2,47,106.360914
Bacia 03,145,312.225035
Bacia 04,122,248.275912
Bacia 05,263,546.051881
Bacia 06,667,1241.007582


##### Exportar excel final

# Resultados

In [93]:
pop_aps = populacao_aps['População 2022'].sum()
print(f"A população total na APS em 2022 é de {pop_aps:.0f}")
dom_aps = len(domparticular_aps)
print(f"Os domicílios totais na APS em 2022 é de {dom_aps:.0f}")

resultado_dompop = bacias_populacao.copy()

resultado_dompop['Dom % bacias'] = resultado_dompop['Domicílios']/(resultado_dompop['Domicílios'].sum())
resultado_dompop['Dom % APS'] = resultado_dompop['Domicílios']/dom_aps
resultado_dompop['Pop % bacias'] = resultado_dompop['População']/(resultado_dompop['População'].sum())
resultado_dompop['Pop % APS'] = resultado_dompop['População']/pop_aps
resultado_dompop['Dom % APS']
display(resultado_dompop)

A população total na APS em 2022 é de 26457
Os domicílios totais na APS em 2022 é de 13439


,Domicílios,População,Dom % bacias,Dom % APS,Pop % bacias,Pop % APS
Name,,,,,,
Bacia 01,55,109.179775,0.004546,0.004093,0.004462,0.004127
Bacia 01.1,25,38.392857,0.002066,0.001860,0.001569,0.001451
Bacia 02,556,1223.547315,0.045958,0.041372,0.050006,0.046246
Bacia 02.1,1179,2630.597045,0.097454,0.087730,0.107512,0.099428
Bacia 02.2,47,106.360914,0.003885,0.003497,0.004347,0.004020
Bacia 03,145,312.225035,0.011985,0.010789,0.012761,0.011801
Bacia 04,122,248.275912,0.010084,0.009078,0.010147,0.009384
Bacia 05,263,546.051881,0.021739,0.019570,0.022317,0.020639
Bacia 06,667,1241.007582,0.055133,0.049632,0.050720,0.046906


In [94]:
tem_agr = int(resultado_dompop['Dom % APS'].sum()*dom_aps)
dom_aps
novdom_aps = int(dom_aps*0.9)
posso_tirar = int(tem_agr - novdom_aps)
print(f'O total é {dom_aps}, agr tenho {tem_agr}. 90% seria {novdom_aps}. Posso tirar {posso_tirar}')

O total é 13439, agr tenho 12098. 90% seria 12095. Posso tirar 3


# Extensão e Área das Bacias

In [95]:
eixolog = gpd.read_file(r'C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul\ServPass+EixoLog (para cálculos).gpkg')
eixolog =eixolog.to_crs(crs)
bacias_area = bacias.to_crs(crs)

bacias_area_len = somar_extensao_polig(eixolog, bacias_area, poly_id_col="Name")
bacias_area_len["Área (km²)"] = bacias_area.geometry.area / 10**6

bacias_area_len = bacias_area_len.set_index("Name")

resultado_final = resultado_dompop.merge(
    bacias_area_len,
    left_index=True,
    right_index=True,
    how="left"
)

resultado_final = resultado_final[['Domicílios','Dom % bacias','Dom % APS','População','Pop % bacias','Pop % APS','Extensão de Rede (m)','Área (km²)']].transpose()

display(resultado_final)

c:\Users\gabriel.coimbra\Documents\GitHub\EngSanitariaAmbiental\venv\Lib\site-packages\geopandas\geoseries.py:860: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has changed compared to previous versions of GeoPandas.
Given a GeoSeries 's', you can use '~s.is_empty & s.notna()' to get back the old behaviour.

To further ignore this warning, you can do: 
import warnings; warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
  return self.notna()


Name,Bacia 01,Bacia 01.1,Bacia 02,Bacia 02.1,Bacia 02.2,Bacia 03,Bacia 04,Bacia 05,Bacia 06,Bacia 07,...,Bacia 16.1,Bacia 17,Bacia 18,Bacia 19,Bacia 20,Bacia 21,Bacia 21.1,Bacia 21.2,Bacia 21.3,Bacia 21.4
Domicílios,55.000000,25.000000,556.000000,1179.000000,47.000000,145.000000,122.000000,263.000000,667.000000,625.000000,...,66.000000,298.000000,23.000000,16.000000,57.000000,198.000000,18.000000,114.000000,42.000000,36.000000
Dom % bacias,0.004546,0.002066,0.045958,0.097454,0.003885,0.011985,0.010084,0.021739,0.055133,0.051661,...,0.005455,0.024632,0.001901,0.001323,0.004712,0.016366,0.001488,0.009423,0.003472,0.002976
Dom % APS,0.004093,0.001860,0.041372,0.087730,0.003497,0.010789,0.009078,0.019570,0.049632,0.046506,...,0.004911,0.022174,0.001711,0.001191,0.004241,0.014733,0.001339,0.008483,0.003125,0.002679
População,109.179775,38.392857,1223.547315,2630.597045,106.360914,312.225035,248.275912,546.051881,1241.007582,1298.465343,...,147.849374,672.114700,52.040730,35.696751,119.436416,441.117231,40.158845,231.927416,60.563707,50.666667
Pop % bacias,0.004462,0.001569,0.050006,0.107512,0.004347,0.012761,0.010147,0.022317,0.050720,0.053068,...,0.006043,0.027469,0.002127,0.001459,0.004881,0.018028,0.001641,0.009479,0.002475,0.002071
Pop % APS,0.004127,0.001451,0.046246,0.099428,0.004020,0.011801,0.009384,0.020639,0.046906,0.049078,...,0.005588,0.025404,0.001967,0.001349,0.004514,0.016673,0.001518,0.008766,0.002289,0.001915
Extensão de Rede (m),2342.260429,401.408592,7162.490655,10935.741938,581.855615,1163.502487,965.304154,2879.387192,5672.395818,5189.047143,...,775.340719,3269.725317,351.585070,257.175479,563.324752,4245.686020,325.221950,1501.349566,1544.328883,374.118908
Área (km²),0.250124,0.126784,0.219317,0.075735,0.422146,0.035952,0.098922,0.021439,0.096988,0.214964,...,0.157540,0.061155,0.230956,0.046658,0.020287,0.033642,0.295966,0.048937,0.018440,0.043958


In [ ]:
resultado_final.to_excel(os.path.join(r'C:\Users\gabriel.coimbra\Desktop\CORSAN\Caçapava do Sul\SaídaCódigo\Param_BH.xlsx'))